# 09. Structured & Record Arrays: Beginner Guide

### 📌 Overview & Architectural Context
Welcome to **09. Structured & Record Arrays**. Structured arrays allow creating C-style struct data buffers in NumPy where each row holds heterogeneous fields (integers, floats, fixed-length strings) packed contiguously in memory. This notebook covers structured `np.dtype` declarations, field access, memory alignment via `align=True` for hardware performance, record arrays with dot-notation access (`np.recarray`), and multi-field sorting.

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Dtype Construction with `np.dtype`
- [x] 🔹 Field Access: `arr['field']`
- [x] 🔹 Record Arrays with Dot Notation (`np.recarray`)
- [x] 🔹 Memory Alignment with `align=True`


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from ../data/raw_transactions.csv (14251 clean aligned rows):
- amounts array: shape (14251,), dtype float64
- fraud_flags array: shape (14251,), dtype int8
- account_ages array: shape (14251,), dtype float32


### 🔹 Dtype Construction with `np.dtype`
- **What it does:** Returns the data type object (dtype) describing the element type and byte order.
- **Syntax:** `ndarray.dtype / Series.dtype`
- **Key Note:** Homogeneous data types ensure maximum SIMD CPU cache alignment.
- **Dataset Application & Code Demonstration:** Applies Dtype Construction with `np.dtype` on fintech records using columns `is_fraud`, `transaction_amount`, `transaction_id` to demonstrate real-world execution.


In [2]:
tx_struct_dtype = np.dtype([
    ('tx_id', 'U12'),
    ('amount', np.float64),
    ('is_fraud', np.int8)
])
struct_records = np.array([
    (clean_raw['transaction_id'][i], clean_raw['transaction_amount'][i], clean_raw['is_fraud'][i])
    for i in range(5)
], dtype=tx_struct_dtype)
print('Structured Transactions Array:\n', struct_records)

Structured Transactions Array:
 [('TX109326',  607.78, 0) ('TX106376', 1819.11, 1)
 ('TX103301',   64.08, 0) ('TX110701', 1025.73, 0)
 ('TX103284',  772.74, 0)]


### 🔹 Field Access: `arr['field']`
- **What it does:** Extracts `amount` and `is_fraud` fields as zero-copy views.
- **Syntax:** `arr['field']`
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Applies Field Access on fintech records using columns `is_fraud` to demonstrate real-world execution.


In [3]:
print('Amount Field View:', struct_records['amount'])
print('Fraud Field View:', struct_records['is_fraud'])

Amount Field View: [ 607.78 1819.11   64.08 1025.73  772.74]
Fraud Field View: [0 1 0 0 0]


### 🔹 Record Arrays with Dot Notation (`np.recarray`)
- **What it does:** Enables dot-notation attribute access (`rec.amount`) over structured memory.
- **Syntax:** `np.recarray`
  - **Parameters:**
    - `row_label` (*hashable*): Row label.
    - `col_label` (*hashable*): Column label.
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Demonstrates Record Arrays with Dot Notation (`np.recarray`) with practical fintech data structures and variables in the following code block.


In [4]:
rec_tx = struct_records.view(np.recarray)
print('Dot Notation Access (rec_tx.tx_id):', rec_tx.tx_id)
print('Dot Notation Access (rec_tx.amount):', rec_tx.amount)

Dot Notation Access (rec_tx.tx_id): ['TX109326' 'TX106376' 'TX103301' 'TX110701' 'TX103284']
Dot Notation Access (rec_tx.amount): [ 607.78 1819.11   64.08 1025.73  772.74]


### 🔹 Memory Alignment with `align=True`
- **What it does:** Pads composite C-structs for 64-bit hardware alignment.
- **Syntax:** `align=True`
- **Key Note:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.
- **Dataset Application & Code Demonstration:** Demonstrates Memory Alignment with `align=True` with practical fintech data structures and variables in the following code block.


In [5]:
unaligned_dt = np.dtype([('flag', 'i1'), ('amt', 'f8')])
aligned_dt = np.dtype([('flag', 'i1'), ('amt', 'f8')], align=True)
print('Unaligned itemsize:', unaligned_dt.itemsize, 'bytes')
print('Aligned itemsize (padded):', aligned_dt.itemsize, 'bytes')

Unaligned itemsize: 9 bytes
Aligned itemsize (padded): 16 bytes


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Filtering Structured Transaction Records
- **Objective:** Q1: Filtering Structured Transaction Records
- **Approach:** Filter structured transaction records where `amount > 200` and `is_fraud == 0`.
- **Syntax:** `struct_records[(struct_records['amount'] > 200) & (struct_records['is_fraud'] == 0)]`

In [6]:
filtered_records = struct_records[(struct_records['amount'] > 100) & (struct_records['is_fraud'] == 0)]
print('Filtered Valid Records:\n', filtered_records)

Filtered Valid Records:
 [('TX109326',  607.78, 0) ('TX110701', 1025.73, 0)
 ('TX103284',  772.74, 0)]
